In [1]:
# point cloud to mesh
# %pip install open3d

#libraries used
import numpy as np
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
#create paths and load data
input_path="Thesis/PointClouds/"
output_path="Thesis/MeshedImages/"
dataname=o3d.io.read_point_cloud("car0.pcd")


In [3]:
print(dataname)


PointCloud with 180205 points.


In [4]:
#used to check if points are accurate

array=np.asarray(dataname.points)

with open("car0.txt", mode='w') as f:
    for i in range(len(array)):
        f.write("%f     "%float(array[i][0].item()))
        f.write("%f     "%float(array[i][1].item()))
        f.write("%f     \n"%float(array[i][2].item()))



In [5]:
#checking but deym its a shark gills xD
o3d.visualization.draw_geometries([dataname])

In [ ]:
#To Mesh tests in Alpha (RIP GPU)
get_meshAlpha = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(dataname,0.03)


In [6]:
#Mesh from point cloud
get_meshAlpha.compute_triangle_normals(normalized=True)
print("Displaying reconstructed mesh ...")
o3d.visualization.draw_geometries([get_meshAlpha], mesh_show_back_face=True)

NameError: name 'get_meshAlpha' is not defined

In [7]:
#Vertex DownSampling 
dataname=o3d.io.read_point_cloud("car1.pcd")
#print("Downsample the point cloud with a voxel of 0.05")
downpcd = dataname.voxel_down_sample(voxel_size=0.05)
o3d.visualization.draw_geometries([downpcd])


In [8]:
#Vertex Normal Estimation
print("Recompute the normal of the downsampled point cloud")
downpcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
o3d.visualization.draw_geometries([downpcd])

Recompute the normal of the downsampled point cloud


In [51]:
#dataname.compute_vertex_normals() NEEDS NORMALS AAAAAHHHH
radii = [0.005, 0.01, 0.02, 0.04]
get_meshBPA = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(dataname, o3d.utility.DoubleVector(radii))

RuntimeError: [Open3D Error] (class std::shared_ptr<class open3d::geometry::TriangleMesh> __cdecl open3d::geometry::BallPivoting::Run(const class std::vector<double,class std::allocator<double> > &)) D:\a\Open3D\Open3D\cpp\open3d\geometry\SurfaceReconstructionBallPivoting.cpp:677: ReconstructBallPivoting requires normals


In [57]:
#USING OPEN 3D DATASET
dataset = o3d.data.EaglePointCloud()
pcd = o3d.io.read_point_cloud(dataset.path)
print(pcd)

PointCloud with 796825 points.


In [58]:
o3d.visualization.draw_geometries([pcd])

In [59]:
#Poisson Surface Reconstruction.....Needs normals
print('run Poisson surface reconstruction')
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=9)
print(mesh)
o3d.visualization.draw_geometries([mesh],)

run Poisson surface reconstruction
[Open3D DEBUG] Input Points / Samples: 796825 / 368254
[Open3D DEBUG] #   Got kernel density: 0.149 (s), 311.211 (MB) / 506.379 (MB) / 896 (MB)
[Open3D DEBUG] #     Got normal field: 0.493 (s), 417.32 (MB) / 506.379 (MB) / 896 (MB)
[Open3D DEBUG] Point weight / Estimated Area: 2.623550e-06 / 2.090510e+00
[Open3D DEBUG] #       Finalized tree: 0.734 (s), 517.086 (MB) / 517.086 (MB) / 896 (MB)
[Open3D DEBUG] #  Set FEM constraints: 1.168 (s), 420.906 (MB) / 517.086 (MB) / 896 (MB)
[Open3D DEBUG] #Set point constraints: 0.348 (s), 393.457 (MB) / 517.086 (MB) / 896 (MB)
[Open3D DEBUG] Leaf Nodes / Active Nodes / Ghost Nodes: 2945433 / 3365000 / 1209
[Open3D DEBUG] Memory Usage: 393.457 MB
[Open3D DEBUG] # Linear system solved: 2.029 (s), 467.055 (MB) / 517.086 (MB) / 896 (MB)
[Open3D DEBUG] Got average: 0.063 (s), 379.559 (MB) / 517.086 (MB) / 896 (MB)
[Open3D DEBUG] Iso-Value: 5.028479e-01 = 4.006817e+05 / 7.968250e+05
[Open3D DEBUG] #          Total Sol